In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

In [2]:
# data import
df = pd.read_csv("C:/Users/USERSS/OneDrive/Documents/Data Science Projects/telco/IBM Telco Customer churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print(df.columns.tolist())

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [4]:
#check missing values
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
print("Original Shape:", df.shape)
print("Missing values in dataset:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())

Original Shape: (7043, 21)
Missing values in dataset: 0
Duplicates: 0


In [7]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan))
df.dropna(inplace=True)
df.drop(columns=['customerID'], inplace=True)

In [8]:
X = df.drop(columns=['Churn'])
y = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

In [9]:
# Balance: Target variable distribution
# Value counts
print("\nTarget Variable Value Counts:")
print(df["Churn"].value_counts())


Target Variable Value Counts:
Churn
No     5163
Yes    1869
Name: count, dtype: int64


In [10]:
# Percentage breakdown of customer churn rate
print("\nTarget Variable Class Distribution (%):")
print(df["Churn"].value_counts(normalize=True) * 100)


Target Variable Class Distribution (%):
Churn
No     73.421502
Yes    26.578498
Name: proportion, dtype: float64


In [11]:
# Check and remove duplicate records
print("Duplicate rows before drop:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

Duplicate rows before drop: 22


In [12]:
# Identify and encode categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)



C:\Users\USERSS\AppData\Local\Temp\ipykernel_3336\3175670986.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns.tolist()


In [13]:
final_features = X_encoded.columns.tolist()
print(f"Features before encoding:{X.shape[1]} | Features after encoding: {X_encoded.shape[1]}")


Features before encoding:19 | Features after encoding: 30


In [14]:
print(df.columns)

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')


In [15]:
# Prepare Data for Modelling
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
# Optional Scaling for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [17]:
# Train Logistic Regression
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train_scaled, y_train)
lr_preds = lr.predict(X_test_scaled)
print("\n--- Logistic Regression ---")
print(f"Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
print(f"Precision: {precision_score(y_test, lr_preds):.4f}")
print(f"Recall: {recall_score(y_test, lr_preds):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, lr_preds))


--- Logistic Regression ---
Accuracy: 0.8038
Precision: 0.6476
Recall: 0.5749
Confusion Matrix:
 [[916 117]
 [159 215]]


In [18]:
# Train Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
print("\n--- Random Forest ---")
print(f"Accuracy: {accuracy_score(y_test, rf_preds):.4f}")
print(f"Precision: {precision_score(y_test, rf_preds):.4f}")
print(f"Recall: {recall_score(y_test, rf_preds):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_preds))


--- Random Forest ---
Accuracy: 0.7896
Precision: 0.6258
Recall: 0.5187
Confusion Matrix:
 [[917 116]
 [180 194]]


In [19]:
# Feature Importance (Random Forest)
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
print("\nTop 5 Features:\n", importances.sort_values(ascending=False).head(5))


Top 5 Features:
 TotalCharges                      0.191435
tenure                            0.171020
MonthlyCharges                    0.168400
InternetService_Fiber optic       0.039481
PaymentMethod_Electronic check    0.037416
dtype: float64


In [20]:
# Part I: Export Models and Features using joblib
joblib.dump(lr, "logistic_regression_model.joblib")
joblib.dump(rf, "random_forest_model.joblib")
joblib.dump(scaler, "scaler.joblib")
joblib.dump(final_features, "feature_names.joblib")

['feature_names.joblib']

In [21]:
print("Model and columns successfully saved!")

Model and columns successfully saved!


In [22]:
app_code =  '''
import streamlit as st
import pandas as pd
import joblib
import numpy as np

# Load artifacts using joblib
@st.cache_resource
def load_artifacts():
    lr_model = joblib.load("logistic_regression_model.joblib")
    rf_model = joblib.load("random_forest_model.joblib")
    scaler = joblib.load("scaler.joblib")
    feature_names = joblib.load("feature_names.joblib")
    return lr_model, rf_model, scaler, feature_names

lr_model, rf_model, scaler, feature_names = load_artifacts()

# App UI
st.title("Telecommunications Customer Churn Predictor")
st.write("This application predicts the likelihood of a customer leaving based on their demographics and service usage.")

model_choice = st.selectbox("Select Prediction Model:", ["Logistic Regression", "Random Forest"])

st.subheader("Customer Demographics")
gender = st.selectbox("Gender", ["Male", "Female"])
senior = st.selectbox("Senior Citizen", ["Yes", "No"])
partner = st.selectbox("Partner", ["Yes", "No"])
dependents = st.selectbox("Dependents", ["Yes", "No"])

st.subheader("Customer Account & Services")
tenure = st.number_input("Tenure (Months)", min_value=0, max_value=100, value=12)
phone_service = st.selectbox("Phone Service", ["Yes", "No"])
multiple_lines = st.selectbox("Multiple Lines", ["No", "Yes", "No phone service"])
internet_service = st.selectbox("Internet Service", ["DSL", "Fiber optic", "No"])
online_security = st.selectbox("Online Security", ["No", "Yes", "No internet service"])
online_backup = st.selectbox("Online Backup", ["No", "Yes", "No internet service"])
device_protection = st.selectbox("Device Protection", ["No", "Yes", "No internet service"])
tech_support = st.selectbox("Tech Support", ["No", "Yes", "No internet service"])
streaming_tv = st.selectbox("Streaming TV", ["No", "Yes", "No internet service"])
streaming_movies = st.selectbox("Streaming Movies", ["No", "Yes", "No internet service"])

st.subheader("Billing Information")
contract = st.selectbox("Contract Type", ["Month-to-month", "One year", "Two year"])
paperless = st.selectbox("Paperless Billing", ["Yes", "No"])
payment_method = st.selectbox("Payment Method", [
    "Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"
])
monthly_charges = st.number_input("Monthly Charges ($)", min_value=0.0, value=50.0)
total_charges = st.number_input("Total Charges ($)", min_value=0.0, value=500.0)

# Prediction Logic
if st.button("Predict Churn"):
    # 1. Create a DataFrame for the input
    input_data = pd.DataFrame([{
        "gender": gender, "SeniorCitizen": 1 if senior == "Yes" else 0, "Partner": partner,
        "Dependents": dependents, "tenure": tenure, "PhoneService": phone_service,
        "MultipleLines": multiple_lines, "InternetService": internet_service,
        "OnlineSecurity": online_security, "OnlineBackup": online_backup,
        "DeviceProtection": device_protection, "TechSupport": tech_support,
        "StreamingTV": streaming_tv, "StreamingMovies": streaming_movies,
        "Contract": contract, "PaperlessBilling": paperless, "PaymentMethod": payment_method,
        "MonthlyCharges": monthly_charges, "TotalCharges": total_charges
    }])
    
    # 2. Encode to match training features
    input_encoded = pd.get_dummies(input_data)
    
    # 3. Align features with saved feature names
    input_aligned = input_encoded.reindex(columns=feature_names, fill_value=0)
    
    # 4. Predict
    if model_choice == "Logistic Regression":
        input_scaled = scaler.transform(input_aligned)
        prediction = lr_model.predict(input_scaled)[0]
        prob = lr_model.predict_proba(input_scaled)[0][1]
    else:
        prediction = rf_model.predict(input_aligned)[0]
        prob = rf_model.predict_proba(input_aligned)[0][1]
        
    st.markdown("---")
    if prediction == 1:
        st.error(f" **Prediction: Customer is likely to CHURN.** (Probability: {prob:.2%})")
    else:
        st.success(f" **Prediction: Customer is likely to REMAIN.** (Probability: {1-prob:.2%})")

st.markdown("---")
st.caption("Disclaimer: This application is for educational purposes and is not a guaranteed business decision or professional assessment.")

'''
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)
print("app.py successfully created!")

app.py successfully created!
